# 🐍 Intro to Python — Exercise Solutions

Worked answers to all five practice problems from the `Intro_Python` notebook.

---

| Exercise | Topic |
|----------|-------|
| 1 | FizzBuzz — conditionals in a loop |
| 2 | Fibonacci — function + iterative sequence |
| 3 | Temperature converter — list comprehension |
| 4 | Damped oscillation — NumPy maths + Matplotlib |
| 5 | Quadratic fit — `np.polyfit` + residual analysis |

In [ ]:
# Imports used across all exercises
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

---
## Exercise 1 — FizzBuzz

**Task:** Write a loop from 1 to 30. Print `"Fizz"` for multiples of 3,
`"Buzz"` for multiples of 5, `"FizzBuzz"` for multiples of **both**, and
the number otherwise.

**Key idea:** Check the combined condition (`% 15 == 0`) *first*, otherwise
a number like 15 would match the `% 3` branch and only print `"Fizz"`.

In [ ]:
for i in range(1, 31):
    if i % 15 == 0:        # divisible by both 3 and 5 — must come first!
        print('FizzBuzz')
    elif i % 3 == 0:
        print('Fizz')
    elif i % 5 == 0:
        print('Buzz')
    else:
        print(i)

---
## Exercise 2 — Fibonacci

**Task:** Write a function `fibonacci(n)` that returns a list of the first
`n` Fibonacci numbers.

The sequence starts `0, 1, 1, 2, 3, 5, 8, 13, …` where each term is the
sum of the two before it.

In [ ]:
def fibonacci(n):
    """Return a list of the first n Fibonacci numbers."""
    if n <= 0:
        return []
    if n == 1:
        return [0]

    seq = [0, 1]
    while len(seq) < n:
        seq.append(seq[-1] + seq[-2])  # next = last + second-to-last
    return seq


# --- Test ---
for n in [1, 5, 10, 15]:
    print(f'fibonacci({n:2d}) → {fibonacci(n)}')

**How it works:**
- Edge cases (`n ≤ 0`, `n == 1`) are handled before the loop.
- We seed the list with `[0, 1]`, then keep appending `seq[-1] + seq[-2]`
  until we have `n` elements.
- Negative indexing (`seq[-1]` = last element, `seq[-2]` = second-to-last)
  keeps the code clean.

---
## Exercise 3 — Temperature Converter

**Task:** Using a list comprehension, convert `[-10, 0, 20, 37, 100]` (°C)
to Fahrenheit.

Formula: $F = \dfrac{9}{5} \cdot C + 32$

In [ ]:
celsius    = [-10, 0, 20, 37, 100]
fahrenheit = [c * 9/5 + 32 for c in celsius]   # list comprehension

print(f"{'Celsius':>10}  {'Fahrenheit':>12}")
print('-' * 25)
for c, f in zip(celsius, fahrenheit):
    print(f"{c:>10}  {f:>11.1f}")

**How it works:**
- The list comprehension `[expr for item in iterable]` builds the converted
  list in a single readable line — equivalent to a `for` loop with `.append()`.
- `zip(celsius, fahrenheit)` pairs the two lists so we can print them
  side-by-side.

---
## Exercise 4 — Damped Oscillation

**Task:** Plot $y = e^{-0.3x} \cdot \cos(2\pi x)$ for $x \in [0, 10]$.

| Component | Role |
|-----------|------|
| $e^{-0.3x}$ | Exponential decay — shrinks the amplitude over time |
| $\cos(2\pi x)$ | Oscillation — one full cycle per unit of $x$ |
| Product | A sinusoid whose amplitude decays toward zero |

In [ ]:
x = np.linspace(0, 10, 500)
y = np.exp(-0.3 * x) * np.cos(2 * np.pi * x)

# Envelope shows the decaying amplitude ceiling/floor
envelope =  np.exp(-0.3 * x)

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(x, y,
        color='steelblue', linewidth=2,
        label=r'$y = e^{-0.3x}\cos(2\pi x)$')
ax.plot(x,  envelope,
        color='tomato', linewidth=1.2, linestyle='--', alpha=0.7,
        label=r'Envelope $\pm\,e^{-0.3x}$')
ax.plot(x, -envelope,
        color='tomato', linewidth=1.2, linestyle='--', alpha=0.7)
ax.fill_between(x, -envelope, envelope, color='tomato', alpha=0.07)
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')

ax.set_title('Damped Oscillation', fontsize=14)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Exercise 5 — Quadratic Fit

**Task:** Generate `x = np.linspace(0, 5, 50)` and
$y = 4x^2 - 3x + 2 + \varepsilon$. Fit a degree-2 polynomial;
compare recovered coefficients to the true values.

`np.polyfit(x, y, deg=2)` returns `[a, b, c]` for $ax^2 + bx + c$,
minimising the sum of squared residuals.

In [ ]:
# --- Generate noisy data ---
rng = np.random.default_rng(42)
x   = np.linspace(0, 5, 50)
y   = 4*x**2 - 3*x + 2 + rng.normal(0, 3, size=len(x))   # true: a=4, b=-3, c=2

# --- Fit degree-2 polynomial ---
coeffs = np.polyfit(x, y, deg=2)
a_fit, b_fit, c_fit = coeffs

# --- Report coefficients ---
print('Coefficient comparison')
print(f"{'':12} {'True':>8}  {'Fitted':>8}  {'Error':>8}")
print('-' * 44)
print(f"{'a  (x²)':12} {4:>8.3f}  {a_fit:>8.3f}  {a_fit - 4:>+8.3f}")
print(f"{'b  (x)':12} {-3:>8.3f}  {b_fit:>8.3f}  {b_fit - (-3):>+8.3f}")
print(f"{'c  (const)':12} {2:>8.3f}  {c_fit:>8.3f}  {c_fit - 2:>+8.3f}")

# --- R² ---
y_hat  = np.polyval(coeffs, x)
ss_res = np.sum((y - y_hat)**2)
ss_tot = np.sum((y - y.mean())**2)
r2     = 1 - ss_res / ss_tot
print(f'\nR² = {r2:.4f}')

In [ ]:
# --- Plot: data, true curve, fitted curve ---
x_smooth    = np.linspace(0, 5, 300)
y_true_line = 4*x_smooth**2 - 3*x_smooth + 2
y_fit_line  = np.polyval(coeffs, x_smooth)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left — data + fits
ax = axes[0]
ax.scatter(x, y, color='steelblue', edgecolors='white',
           s=55, zorder=3, label='Noisy data')
ax.plot(x_smooth, y_true_line,
        color='black', linewidth=1.5, linestyle='--',
        label='True: $4x^2 - 3x + 2$')
ax.plot(x_smooth, y_fit_line,
        color='tomato', linewidth=2.5,
        label=f'Fitted: ${a_fit:.2f}x^2 {b_fit:+.2f}x {c_fit:+.2f}$\n$R^2={r2:.3f}$')
ax.set_title('Quadratic Regression')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Right — residuals
ax = axes[1]
residuals = y - np.polyval(coeffs, x)
ax.scatter(x, residuals, color='mediumpurple',
           edgecolors='white', s=55, zorder=3)
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Residuals  (y − ŷ)')
ax.set_xlabel('x')
ax.set_ylabel('Residual')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation:**
- The fitted coefficients should be close to the true values **a = 4, b = −3, c = 2**.
  Small deviations are expected — the noise ($\sigma = 3$) shifts the estimates slightly.
- The **R²** value close to 1 confirms the quadratic model explains most of the variance.
- The **residual plot** (right) should look like random scatter around zero — no obvious
  pattern means the model is well-specified.

---
*End of solutions* 🐍